# Kata 08 — Memoria Jerárquica con `CLAUDE.md`

> Spec: [`specs/008-claude-md-memory/spec.md`](../../specs/008-claude-md-memory/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

## Setup

Si `ANTHROPIC_API_KEY` no está en el entorno, se pedirá aquí (sin echo).

In [ ]:
from katas._shared.bootstrap import bootstrap

client, settings = bootstrap(budget_calls=6)
print("modelo:", settings.model)

## 1. Concepto en mis palabras

`CLAUDE.md` es la memoria persistente del agente. Tres niveles cargados en cascada:

1. `~/.claude/CLAUDE.md` — preferencias del usuario.
2. `<repo>/CLAUDE.md` — convenciones del equipo.
3. `<repo>/<subpath>/CLAUDE.md` — reglas específicas del módulo.

En precedencia: más específico gana. Lo modular se compone con `@imports`.

## 2. Por qué importa

Repetir convenciones en cada prompt cuesta tokens y diverge. La cascada centraliza una sola fuente de verdad por nivel y se carga automáticamente. En la API directa lo emulo concatenando los archivos en orden de precedencia y entregándolo como `system=`.

## 3. Ejemplo correcto — cascada y precedencia

In [ ]:
USER_MD = '''# Mis preferencias
- Respuestas breves, sin listas largas.
- Prefiero TypeScript sobre JavaScript.'''

PROJECT_MD = '''# Convenciones del proyecto
- Lenguaje del proyecto: Python 3.11.
- Tests con pytest.
- Estilo: ruff, no black.

@docs/architecture.md
'''

ARCHITECTURE_MD = '''## Arquitectura
- Capas: api → service → repo.
- Sin lógica en views.'''

MODULE_MD = '''# src/auth/CLAUDE.md
- Toda contraseña pasa por `secrets.compare_digest`.
- Sesiones expiran a los 30 min.'''

import re

def resolve_imports(text: str, files: dict) -> str:
    def repl(m):
        path = m.group(1)
        return files.get(path, f"<MISSING:{path}>")
    return re.sub(r"^@(\S+)\s*$", repl, text, flags=re.MULTILINE)

def merge_claude_md(*layers, files):
    parts = []
    for layer_name, layer_text in layers:
        if layer_text:
            resolved = resolve_imports(layer_text, files)
            parts.append(f"<!-- from: {layer_name} -->\n{resolved}")
    return "\n\n".join(parts)

FILES = {"docs/architecture.md": ARCHITECTURE_MD}
merged = merge_claude_md(
    ("user", USER_MD),
    ("project", PROJECT_MD),
    ("module:src/auth", MODULE_MD),
    files=FILES,
)
print(merged)

### 3.1 El system prompt es el merge

In [ ]:
resp = client.messages.create(
    system=merged,
    messages=[{"role": "user", "content": "¿Qué stack uso para autenticar usuarios en este proyecto?"}],
)
print(next((b.text for b in resp.content if b.type == "text"), ""))

El modelo combina:

- "Python 3.11" (proyecto) — gana sobre "TypeScript" (preferencia personal) porque más específico.
- "secrets.compare_digest" + "30 min sesión" del módulo `src/auth`.

Sin que yo repita ninguna de esas líneas en cada turno.

## 4. Anti-patrón — todo embutido en cada turno

In [ ]:
messy_prompt = (
    "Eres un asistente. Recuerda: este proyecto usa Python 3.11. Tests con pytest. Estilo ruff. "
    "Capas: api → service → repo. Auth con secrets.compare_digest. Sesiones 30 min. "
    "Mis preferencias: respuestas breves, TypeScript. La arquitectura es: ... "
    "Por favor responde en estilo casual. Recuerda Python 3.11. ¿Qué stack uso para autenticar?"
)
resp_bad = client.messages.create(
    system="Eres un asistente.",
    messages=[{"role": "user", "content": messy_prompt}],
)
print("respuesta:", next((b.text for b in resp_bad.content if b.type == "text"), "")[:300])
print("\ninput_tokens:", resp_bad.usage.input_tokens)

Problemas del anti-patrón:

- Las reglas están en el `user` content, así no se cachean entre turnos (rompe Kata 10).
- Mezcla preferencias personales con reglas de equipo (las personales contaminarán al equipo).
- Repetir "Python 3.11" varias veces en el mismo prompt no mejora la atención; la dispersa (Kata 11).

## 5. Argumento de certificación

- **Tres niveles, un orden**: usuario → proyecto → módulo. Más específico gana.
- **`@imports` mantienen el archivo principal corto** y permiten reutilizar `architecture.md` en varios proyectos.
- **El merge va en `system=`**, no en cada `user` turn → estable, cacheable.
- **Conexión con otros katas**: parejas naturales con Kata 9 (reglas por path), Kata 10 (caché por prefijo estable) y Kata 11 (no enterrar reglas).

## 6. Auto-evaluación

**1. ¿Dónde guardo "este equipo usa pytest, no unittest"?**

En el `CLAUDE.md` del proyecto (`<repo>/CLAUDE.md`). Convención del equipo, viaja con el repo.

**2. ¿Dónde guardo "yo prefiero respuestas concisas"?**

En `~/.claude/CLAUDE.md`. Es preferencia personal y no debe contaminar al equipo.

**3. ¿Qué ocurre si el archivo del usuario y el del proyecto se contradicen?**

Gana el del proyecto cuando se trata de algo del proyecto (lenguaje, lints, arquitectura). Gana el del usuario cuando es estilo conversacional. La resolución vive en `merge_claude_md`: el orden de los `layers=` en la llamada define la precedencia, y los layers más específicos van después para que su contenido aparezca al final del system prompt (zona de mayor atención, ver Kata 11).